# Amplitude reconstruction and CDF-like output

This notebook starts from notebook 01's artifact, runs the three registered models, exports one CDF-like file per model, and compares their structure and amplitudes with the matching NASA reference.

**Before running:** finish notebook 01 first, then open this notebook from the repository root.

**Result:** three model predictions and three CDF-like exports plotted on calibrated frequency and virtual-height axes. The comparison checks compatibility; predicted amplitudes are not expected to equal measured NASA amplitudes.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib

_interactive = False
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
    _interactive = 'agg' not in str(matplotlib.get_backend()).lower()
except NameError:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt


def show_plot():
    if _interactive and 'agg' not in str(matplotlib.get_backend()).lower():
        plt.show()
    plt.close()

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'pyproject.toml').is_file():
        ROOT = candidate
        break
else:
    raise RuntimeError('Open this notebook from the repository root')
sys.path[:0] = [str(ROOT), str(ROOT / 'src')]
artifact_path = ROOT / 'outputs/notebooks/01_scan.npz'
model_config_path = ROOT / 'configs/model_candidates.json'
reference_cdf_path = ROOT / 'data/samples/i2_av_ksh_1972322002235_v01.cdf'
assert artifact_path.is_file(), 'Run notebook 01 before this notebook'
assert model_config_path.is_file()
assert reference_cdf_path.is_file()

import importlib

from isis_research import ionogram
from isis_research.nasa.cdf_compare import compare_cdf_content
from scripts.pipeline import infer_isis_model

infer_isis_model = importlib.reload(infer_isis_model)
candidate_checkpoint = infer_isis_model.candidate_checkpoint
infer = infer_isis_model.infer
load_model_candidates = infer_isis_model.load_model_candidates
from isis_research.nasa.model_cdf import export_model_cdf, header_from_csa

scan = ionogram.read_validated(artifact_path)
print(scan.intensity.shape, scan.frequency_mhz[[0, -1]], scan.virtual_height_km[[0, -1]])
signal = 1.0 - scan.intensity
plt.figure(figsize=(12, 7))
plt.imshow(
    signal,
    cmap='magma',
    aspect='auto',
    origin='upper',
    extent=[float(scan.frequency_mhz[0]), float(scan.frequency_mhz[-1]), float(scan.virtual_height_km[-1]), float(scan.virtual_height_km[0])],
)
plt.title('Signal-positive calibrated CSA input')
plt.xlabel('frequency (MHz)')
plt.ylabel('virtual height (km)')
show_plot()

## Model prediction

The model receives the calibrated film signal and writes a prediction on the same axes. The validity mask is carried forward; it is not measured amplitude.

In [ ]:
candidate_config = load_model_candidates(model_config_path)
prediction_paths = {}
predictions = {}
for model_name in candidate_config['models']:
    checkpoint = candidate_checkpoint(model_name, model_config_path)
    prediction_path = ROOT / f'outputs/notebooks/01_prediction_{model_name}.npz'
    predictions[model_name] = infer(
        artifact_path,
        checkpoint,
        prediction_path,
        model_name=model_name,
        model_config=model_config_path,
    )
    prediction_paths[model_name] = prediction_path

fig, axes = plt.subplots(1, len(predictions), figsize=(18, 6), constrained_layout=True)
for axis, (model_name, prediction) in zip(axes, predictions.items()):
    axis.imshow(
        prediction,
        cmap='magma',
        aspect='auto',
        vmin=0,
        vmax=1,
        origin='upper',
        extent=[float(scan.frequency_mhz[0]), float(scan.frequency_mhz[-1]), float(scan.virtual_height_km[-1]), float(scan.virtual_height_km[0])],
    )
    axis.set_title(candidate_config['models'][model_name]['label'])
    axis.set_xlabel('Frequency (MHz)')
    axis.set_ylabel('Virtual height (km)')
fig.suptitle('Three calibrated model predictions')
show_plot()

## CDF-like export and reference comparison

Observation time comes from the verified pair name. Fields unavailable from the CSA image are written as explicit unknowns. The comparison reports shared and file-specific variables, then resamples NASA amplitude onto the model grid.

In [ ]:
pair_name = reference_cdf_path.stem
header = header_from_csa(pair_name, 'KSH', scan.frequency_mhz, scan.virtual_height_km)
for model_name, prediction_path in prediction_paths.items():
    cdf_path = ROOT / f'outputs/notebooks/01_model_{model_name}.cdf'
    values, provenance = export_model_cdf(prediction_path, header, cdf_path)
    comparison = compare_cdf_content(reference_cdf_path, cdf_path)
    print(json.dumps({
        'model': model_name,
        'output': str(cdf_path),
        'ampl_shape': list(values['ampl'].shape),
        'reference_cdf': reference_cdf_path.name,
        'shared_variable_count': len(comparison['shared_variables']),
        'only_in_reference': comparison['only_in_nasa'],
        'only_in_model': comparison['only_in_model'],
        'provenance': provenance,
    }, indent=2))

## Visual NASA-versus-model comparison

NASA amplitude is resampled onto each model grid before comparison. Metrics use only pixels valid in both files.

In [ ]:
import cdflib
import numpy as np
from IPython.display import display

from isis_research.nasa.cdf import cdf_amplitude


def read_model_cdf(path):
    cdf = cdflib.CDF(str(path))
    amplitude = np.asarray(cdf.varget('ampl'), dtype=float).T / 255.0
    frequency = np.asarray(cdf.varget('freq'), dtype=float).ravel()
    height = np.asarray(cdf.varget('v_height'), dtype=float).ravel()
    valid = np.asarray(cdf.varget('valid_mask'), dtype=bool).T
    return amplitude, valid, frequency, height

comparison_rows = []
comparison_figures = {}

for model_name, prediction_path in prediction_paths.items():
    cdf_path = ROOT / f'outputs/notebooks/01_model_{model_name}.cdf'
    model_amplitude, model_valid, model_frequency, model_height = read_model_cdf(cdf_path)
    nasa_amplitude_raw, nasa_valid = cdf_amplitude(
        reference_cdf_path, model_frequency, model_height
    )
    nasa_amplitude = np.clip(nasa_amplitude_raw / 255.0, 0.0, 1.0)
    overlap = (
        model_valid
        & nasa_valid
        & np.isfinite(model_amplitude)
        & np.isfinite(nasa_amplitude)
    )
    if not np.any(overlap):
        raise ValueError(f'{model_name}: NASA and model CDFs have no overlapping valid pixels')
    residual = model_amplitude - nasa_amplitude
    overlap_values = residual[overlap]
    nasa_values = nasa_amplitude[overlap]
    model_values = model_amplitude[overlap]
    correlation = (
        float(np.corrcoef(nasa_values, model_values)[0, 1])
        if len(overlap_values) > 1
        and np.std(nasa_values) > 0
        and np.std(model_values) > 0
        else 0.0
    )
    comparison_rows.append({
        'model': model_name,
        'grid_shape': list(model_amplitude.shape),
        'overlap_fraction': float(np.mean(overlap)),
        'mae': float(np.mean(np.abs(overlap_values))),
        'rmse': float(np.sqrt(np.mean(overlap_values ** 2))),
        'correlation': correlation,
    })

    extent = [
        float(model_frequency[0]),
        float(model_frequency[-1]),
        float(model_height[-1]),
        float(model_height[0]),
    ]
    figure, axes = plt.subplots(1, 3, figsize=(18, 5.5), constrained_layout=True)
    images = [
        (nasa_amplitude, 'NASA CDF'),
        (model_amplitude, f'{model_name} model CDF'),
        (np.abs(residual), 'Absolute error'),
    ]
    for axis, (image, title) in zip(axes, images):
        plot = axis.imshow(
            image,
            origin='upper',
            aspect='auto',
            extent=extent,
            vmin=0.0,
            vmax=1.0,
            cmap='viridis',
        )
        axis.set_title(title)
        axis.set_xlabel('Frequency (MHz)')
        axis.set_ylabel('Virtual height (km)')
        figure.colorbar(plot, ax=axis, fraction=0.046, pad=0.04)
    figure.suptitle(
        f'{model_name}: NASA reference vs model output | '
        f'MAE={comparison_rows[-1]["mae"]:.4f}, '
        f'RMSE={comparison_rows[-1]["rmse"]:.4f}, '
        f'corr={comparison_rows[-1]["correlation"]:.3f}'
    )
    figure_path = ROOT / f'outputs/notebooks/03_cdf_compare_{model_name}.png'
    figure.savefig(figure_path, dpi=150)
    comparison_figures[model_name] = str(figure_path)
    display(figure)
    plt.close(figure)

print(json.dumps({
    'reference_cdf': str(reference_cdf_path),
    'comparisons': comparison_rows,
    'figures': comparison_figures,
}, indent=2))